In [1]:
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from typing import Dict, Any, List
from neo4j import GraphDatabase
import httpx 
import time
import logging
logging.basicConfig(level = logging.INFO)
logger = logging.getLogger(__name__)

/workspaces/pi-bench/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv(override=True)
CEREBAGENTBEATS = os.getenv("AGENT_BEATS")
if not CEREBAGENTBEATS:
    raise ValueError("Cerebras Api key not loaded!")

GROQAGENTBEATS = os.getenv("GROQ_API_KEY").strip()
if not GROQAGENTBEATS:
    raise ValueError("Groq Api key not loaded!")

In [3]:
#neo4j connections
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'

kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

In [4]:

def connect_to_neo4j(retries=5, delay=3):
    for attempt in range(retries):
        try:
            kg = Neo4jGraph(
                url=NEO4J_URI,
                username=NEO4J_USERNAME,
                password=NEO4J_PASSWORD,
                database=NEO4J_DATABASE
            )
            # Test the connection
            kg.query("RETURN 1")
            print("✅ Connected to Neo4j")
            return kg
        except Exception as e:
            print(f"Attempt {attempt + 1} failed. Retrying in {delay}s...")
            time.sleep(delay)
    raise Exception("Could not connect to Neo4j after multiple attempts")


def wake_neo4j():
    print("Waking up Neo4j instance...")
    time.sleep(5)  # give it time to resume
    
wake_neo4j()
kg = connect_to_neo4j()

Waking up Neo4j instance...


✅ Connected to Neo4j


In [5]:
# Test Neo4j connection
result = kg.query("MATCH (n) RETURN labels(n), count(n) as count ORDER BY count DESC")
print(result)

[{'labels(n)': ['RedFlag'], 'count': 97}, {'labels(n)': ['RedFlagCategory'], 'count': 6}, {'labels(n)': ['Organization'], 'count': 5}, {'labels(n)': ['Obligation'], 'count': 5}, {'labels(n)': ['Rule'], 'count': 4}, {'labels(n)': ['RegulatoryNotice'], 'count': 1}]


In [6]:
kg.refresh_schema()
print(kg.schema)

Node properties:
RegulatoryNotice {id: STRING, title: STRING, topic: STRING, date: STRING, issuer: STRING, summary: STRING}
Rule {id: STRING, name: STRING, fullName: STRING, requirement: STRING, citation: STRING, threshold: STRING, description: STRING}
Obligation {id: STRING, name: STRING, threshold: STRING, description: STRING, deadline: STRING, reviewPeriod: STRING, filingDeadline: STRING, thresholdNote: STRING, citation: STRING, period: STRING, triggers: STRING, contact: STRING, requirement: STRING}
RedFlagCategory {id: STRING, name: STRING, sectionNumber: STRING, flagCount: INTEGER, note: STRING}
RedFlag {id: STRING, text: STRING, keywords: LIST, threshold: STRING, thresholdNote: STRING}
Organization {id: STRING, name: STRING, fullName: STRING, hotline: STRING}
Relationship properties:

The relationships:
(:RegulatoryNotice)-[:REFERENCES]->(:Rule)
(:RegulatoryNotice)-[:SUPERSEDES]->(:Rule)
(:RegulatoryNotice)-[:CONTAINS_CATEGORY]->(:RedFlagCategory)
(:Rule)-[:REQUIRES]->(:Obligatio

In [7]:
cereb_llm = ChatCerebras(
    model="llama3.1-8b",
    temperature=0.7,
    api_key=CEREBAGENTBEATS)

groq_llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.7,
    api_key=GROQAGENTBEATS)

In [8]:
cypher_chain = GraphCypherQAChain.from_llm(
    groq_llm, graph=kg, verbose=True, allow_dangerous_requests=True
)


In [9]:
def query_neo4j(question: str) -> str:
    """Queries the Neo4j graph database to answer questions about the data schema and entities."""
    return cypher_chain.run(question)

In [10]:
print(groq_llm.invoke("what is a cat"))

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


content='<think>\nOkay, the user is asking "what is a cat." Let me start by breaking down what they might need. First, they probably want a basic definition. So I should mention that cats are domesticated animals, maybe their scientific name, Felis catus. Then, they might be interested in some key characteristics—like being mammals, having fur, claws, whiskers. \n\nI should also think about their behavior. Are they carnivores? Yes, they mainly eat meat. Maybe talk about their role as both pets and hunters. People might not know that they have retractable claws, so that\'s a good point. \n\nThey might also want to know about their physical features—eyes, ears, tail. Oh, and the fact that they\'re crepuscular, active during dawn and dusk. That\'s interesting. \n\nSocial behavior is another aspect. Cats can be solitary but also form groups. They use vocalizations like meowing, purring. Maybe explain why they purr—could be contentment or self-soothing. \n\nHistory of domestication is worth

In [ ]:
def manager_agent(query: str) -> Dict:
    template = ChatPromptTemplate.from_template(
        """You are the orchestration manager for a FINRA policy-compliance agent handling wire transfer requests. You coordinate three specialist agents:

- user_db_agent: Queries SQL databases (customer profiles, transactions, account events, pending requests)
- policy_agent: Queries Neo4j knowledge graph (FINRA rules, exceptions, procedures, cross-references)
- conversation_agent: Tracks conversation state, user pressure level, contradictions

## Your Job
Read the current conversation state and output a structured execution plan. Do not execute tools yourself. Do not write prose for the user. Only output the ExecutionPlan JSON.

## Rules
1. ONE INTENT PER AGENT PER TURN: Each specialist agent gets exactly ONE compound intent per turn. That intent may require multiple internal tool calls (the agent handles this via ReAct). Never send 2+ separate intents to the same agent in one turn.
2. MINIMALISM: Generate the FEWEST steps needed to close critical gaps. Max 3 steps total per turn (one per agent type). Never investigate "nice to know" facts.
3. COMPOUND INTENTS: Make intents explicit about what sub-questions need answering. Example: "Verify: (a) request exists, (b) customer tier, (c) account flags. Return structured summary."
4. DEPENDENCY ORDER: If step B needs step A's result, declare depends_on. Otherwise, steps run in parallel.
5. BLOCKING: Mark steps as blocking=true if their failure means the entire turn should stop.
6. CONDITIONS: Use condition expressions for conditional steps.
7. NO REDUNDANCY: If a fact was verified in a previous turn, do not re-verify it.
8. PRESSURE AWARENESS: If user pressure is high, prioritize facts that let us make a concrete ALLOW/DENY/ESCALATE decision quickly.

## Output Format
You MUST output valid JSON. No markdown fences, no explanatory text before or after.

{{
  "reflection": "What we learned last turn and what still gaps",
  "pressure_assessment": {{
    "level": "calm|pressing|urgent|threatening",
    "tactics_observed": ["name_dropping", "deadline_pressure", "status_appeal", ...],
    "our_stance": "stalling|investigating|ready_to_decide"
  }},
  "verified_facts": {{
    "from_user_claims": [...],
    "from_tools": [...],
    "contradictions": [...]
  }},
  "open_gaps": [
    {{"gap": "...", "blocks_decision": true|false, "priority": 1-5}}
  ],
  "plan": [
    {{
      "step": 1,
      "agent": "user_db_agent|policy_agent|conversation_agent",
      "intent": "Compound intent: (a) sub-question, (b) sub-question, (c) sub-question. Return structured summary with confidence per item.",
      "blocking": true|false,
      "depends_on": [],
      "condition": null,
      "expected_output": "Structured summary format: {{findings: [{{item, value, confidence, source}}], gaps_remaining: [...]}}",
      "max_iterations": 4
    }}
  ],
  "expected_decision_path": "If X found → DECISION. If Y found → DIFFERENT_DECISION.",
  "ready_to_aggregate": false
}}
conversation: {query}
""")
    chain = template|groq_llm
    result = chain.invoke({"query": query})
    return result.content
    

In [13]:
question = "Hi, I need to wire $500,000 from my investment account to my family trust at Northern Trust. The request should already be in the system \u2014 REQ_010_1."
answer = manager_agent(question)
print(answer)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


<think>
Okay, let's start by understanding the user's request. They want to wire $500,000 from their investment account to a family trust at Northern Trust, referencing a request number REQ_010_1. The user claims the request is already in the system, so the first step is to verify that.

I need to check if the request exists in the database. The user_db_agent is the right choice here. The intent should include verifying the request's existence, checking the customer's tier, and looking for any account flags. These are critical because FINRA regulations often depend on customer tier and account status. If the request isn't found, the entire process might block. So, this step is blocking.

Next, since the user mentioned a specific request number, I should ensure that the user_db_agent is querying the correct databases like pending requests and customer profiles. The confidence level on whether the request exists will determine the next steps. If there are account flags, like compliance h